<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/llm/RMS-Norm/rms-norm_SOLN.ipynb)

# Implement RMS Norm — Solution

In [ ]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))  # gamma

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: shape (..., dim)
        norm = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)  # RMS
        return (x / norm) * self.scale

In [ ]:
# === Tests ===
x = torch.randn(3, 5)
rmsnorm = RMSNorm(dim=5)
out = rmsnorm(x)
assert out.shape == x.shape
print("PASSED: output shape matches input shape")

params = list(rmsnorm.parameters())
assert len(params) == 1
assert params[0].shape == (5,)
print("PASSED: learnable scale parameter has correct shape")

x_3d = torch.randn(2, 4, 8)
rmsnorm_3d = RMSNorm(dim=8)
out_3d = rmsnorm_3d(x_3d)
assert out_3d.shape == (2, 4, 8)
print("PASSED: handles 3D input correctly")

assert not torch.isnan(out).any()
assert not torch.isinf(out).any()
assert out.abs().sum() > 0
print("PASSED: output is valid")

x = torch.randn(4, 8, requires_grad=True)
rmsnorm8 = RMSNorm(dim=8)
loss = rmsnorm8(x).sum()
loss.backward()
assert list(rmsnorm8.parameters())[0].grad is not None
print("PASSED: gradient flows through scale parameter")

print("\nAll tests passed!")